# Entrenar la voz propia con Piper (IureVoz, fase T5)

Afina **Piper medium** partiendo del checkpoint mexicano `es_MX-ald` (*warmstart*). Los datos son el corpus sintético que generó `generar_corpus.ipynb` más las grabaciones reales del usuario, repetidas para darles más peso. La pronunciación la decide espeak-ng `es-419`, igual que en la app.

**Entradas** (*Add data*; pueden quedar en subcarpetas, el cuaderno las busca):
- La **salida del cuaderno de generación**: `salida/metadata.csv` y `salida/wavs/`. Si se generó en varias versiones, basta con la última.
- El dataset con las **grabaciones reales**: `mi-voz/metadata.csv` y `mi-voz/NNN.wav`.
- Para **continuar** un entrenamiento cortado, la salida de la versión anterior de *este* cuaderno (`entreno/…/last.ckpt`).

**Kaggle**: *GPU T4 x2* (usa una), *Internet: on* y **Save Version → Save & Run All**. El entrenamiento se detiene solo a las `HORAS_MAX` y deja `last.ckpt` para seguir en otra versión.

**Salida**: ONNX exportados (último y mejor por `val_mel`) con su `.onnx.json`, muestras de audio para escuchar y los checkpoints para continuar. La app la empaqueta en local con `bench/empaquetar_voz.py`.

Todo es **privado**: es la voz del usuario.

In [ ]:
import os, glob, shutil, subprocess, sys, csv, random
from pathlib import Path

NOMBRE = "iurevoz"            # la voz queda como es_MX-<NOMBRE>-medium
LOTE = 16                     # 16 cabe en una T4 (16 GB) en fp32; súbelo si sobra memoria
EPOCAS = 200                  # duración del plan de learning rate (mantenla igual al continuar)
HORAS_MAX = 11                # se detiene solo antes del límite de 12 h de Kaggle
PESO_REAL = 3                 # cuántas veces se repite cada grabación real
SR = 22050

EN_KAGGLE = Path("/kaggle").exists()
RAIZ = Path("/kaggle/working") if EN_KAGGLE else Path("/content")
BUSCAR = "/kaggle/input" if EN_KAGGLE else "/content/drive/MyDrive"
if not EN_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")

def buscar(patron):
    return sorted(glob.glob(f"{BUSCAR}/**/{patron}", recursive=True))

corpus = [Path(p).parent for p in buscar("salida/metadata.csv")]
reales = [Path(p).parent for p in buscar("mi-voz/metadata.csv")]
previos = buscar("entreno/**/last.ckpt")
print("corpus:", corpus, "\nreales:", reales, "\ncontinuar desde:", previos)
assert corpus, "falta la salida del cuaderno de generación (salida/metadata.csv)"

In [ ]:
# Piper 1.8.0 (con la corrección del learning rate) y su espeak-ng; tarda unos minutos en compilar
!apt-get -qq update && apt-get -qq install -y build-essential cmake ninja-build > /dev/null
!git clone -q --depth 1 --branch v1.8.0 https://github.com/OHF-voice/piper1-gpl.git /tmp/piper1-gpl
!cd /tmp/piper1-gpl && pip -q install -e '.[train]' && pip -q install scikit-build cmake ninja onnxscript && ./build_monotonic_align.sh && python setup.py build_ext --inplace > /tmp/piper-build.log 2>&1 && echo "piper listo"

from pathlib import Path

# Lanzador propio, por dos fallas de Piper 1.8.0:
# - sus checkpoints por defecto también vigilan "val_mos" (UTMOS), que necesita torchaudio, y
#   torchaudio ya no tiene paquetes para PyTorch reciente: Lightning falla al final de cada época.
#   Aquí se guardan los 3 mejores por val_mel y el último.
# - aplica el warmstart en on_fit_start, DESPUÉS de restaurar --ckpt_path: al continuar una sesión
#   borraría lo entrenado con el modelo base (y Lightning recupera warmstart_ckpt de los hparams
#   aunque no se pase). Al continuar, este modelo lo omite.
Path("/tmp/piper1-gpl/entrenar_iurevoz.py").write_text('''
import logging
from lightning.pytorch.callbacks import ModelCheckpoint
from piper.train.__main__ import VitsLightningCLI
from piper.train.vits.dataset import VitsDataModule
from piper.train.vits.lightning import VitsModel

class Modelo(VitsModel):
    def on_fit_start(self):
        if self.trainer.ckpt_path:
            logging.info("continuando desde %s: sin warmstart", self.trainer.ckpt_path)
            self._warmstart_ckpt = self._vocoder_warmstart_ckpt = None
        super().on_fit_start()

logging.basicConfig(level=logging.INFO)
VitsLightningCLI(Modelo, VitsDataModule, trainer_defaults={"max_epochs": -1, "callbacks": [
    ModelCheckpoint(monitor="val_mel", mode="min", save_top_k=3, save_last=True,
                    filename="epoch={epoch}-val_mel={val_mel:.4f}", auto_insert_metric_name=False)]})
''')

# El exportador nuevo de ONNX (dynamo), que PyTorch usa por omisión desde la 2.9, no puede con el
# VITS de Piper: se fuerza el clásico (TorchScript).
Path("/tmp/piper1-gpl/exportar_iurevoz.py").write_text('''
import runpy
import torch

_exportar = torch.onnx.export
def exportar(*args, **kwargs):
    kwargs.setdefault("dynamo", False)
    return _exportar(*args, **kwargs)
torch.onnx.export = exportar
runpy.run_module("piper.train.export_onnx", run_name="__main__")
''')

In [ ]:
# Datos: todo a 22050 Hz mono, mismo volumen (RMS) y sin saturar
import numpy as np, soundfile as sf, librosa

DATOS = RAIZ / "datos"
(DATOS / "wavs").mkdir(parents=True, exist_ok=True)

def preparar(origen, destino):
    if destino.exists():
        return True
    x, _ = librosa.load(origen, sr=SR, mono=True)
    rms = np.sqrt(np.mean(x ** 2)) + 1e-9
    x = x * (10 ** (-20 / 20) / rms)          # -20 dBFS RMS
    pico = np.abs(x).max()
    if pico > 0.95:
        x = x * 0.95 / pico
    sf.write(destino, x, SR, subtype="PCM_16")
    return True

filas = []
for c in corpus:
    for l in (c / "metadata.csv").read_text(encoding="utf-8").splitlines():
        i, t = l.split("|", 1)
        if preparar(c / "wavs" / f"{i}.wav", DATOS / "wavs" / f"s{i}.wav"):
            filas.append((f"s{i}.wav", t))
n_sint = len(filas)
for r in reales:
    for l in (r / "metadata.csv").read_text(encoding="utf-8").splitlines():
        i, t = l.split("|", 1)
        preparar(r / f"{i}.wav", DATOS / "wavs" / f"r{i}.wav")
        filas += [(f"r{i}.wav", t)] * PESO_REAL
random.Random(1).shuffle(filas)
with (DATOS / "metadata.csv").open("w", encoding="utf-8", newline="") as f:
    csv.writer(f, delimiter="|").writerows(filas)
print(f"{n_sint} frases sintéticas + {len(filas) - n_sint} reales (x{PESO_REAL}) = {len(filas)}")

In [ ]:
# Checkpoint base: es_MX-ald medium (licencia del dataset: Unlicense)
from huggingface_hub import hf_hub_download
BASE = hf_hub_download("rhasspy/piper-checkpoints", "es/es_MX/ald/medium/epoch=9999-step=1753600.ckpt", repo_type="dataset")

ENTRENO = RAIZ / "entreno"
ENTRENO.mkdir(exist_ok=True)
CONFIG = ENTRENO / f"es_MX-{NOMBRE}-medium.onnx.json"
reanudar = []
if previos:  # continuar: copia el último checkpoint de la versión anterior
    shutil.copy(previos[-1], ENTRENO / "continuar.ckpt")
    reanudar = ["--ckpt_path", str(ENTRENO / "continuar.ckpt")]  # el lanzador omite el warmstart
    for p in buscar("entreno/*.onnx.json")[:1]:
        shutil.copy(p, CONFIG)
print("base:", BASE, "| reanudar:", bool(reanudar))

In [ ]:
cmd = [sys.executable, "entrenar_iurevoz.py", "fit",
    "--data.voice_name", NOMBRE,
    "--data.csv_path", str(DATOS / "metadata.csv"),
    "--data.audio_dir", str(DATOS / "wavs"),
    "--model.sample_rate", str(SR),
    "--data.espeak_voice", "es-419",
    "--data.cache_dir", str(RAIZ / "cache"),
    "--data.config_path", str(CONFIG),
    "--data.batch_size", str(LOTE),
    "--model.warmstart_ckpt", BASE,
    "--model.mos_metric", "none",
    "--trainer.max_epochs", str(EPOCAS),
    "--trainer.max_time", f"00:{HORAS_MAX:02d}:00:00",
    "--trainer.accelerator", "gpu", "--trainer.devices", "1",
    "--trainer.default_root_dir", str(ENTRENO),
] + reanudar
print(" ".join(cmd))
p = subprocess.Popen(cmd, cwd="/tmp/piper1-gpl", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linea in p.stdout:  # solo lo útil: épocas, pérdidas y errores
    if any(k in linea for k in ("Epoch", "epoch", "val_", "Error", "error", "Traceback", "Saving", "max_time")):
        print(linea.rstrip()[-300:], flush=True)
print("código de salida:", p.wait())

In [ ]:
# Exportar a ONNX: el último y el mejor por val_mel. El mel se estanca pronto y los artefactos se
# siguen quitando después, así que el último suele sonar mejor: se elige escuchando.
ckpts = {"ultimo": max(glob.glob(f"{ENTRENO}/**/last.ckpt", recursive=True), key=os.path.getmtime)}
mel = glob.glob(f"{ENTRENO}/**/*val_mel=*.ckpt", recursive=True)
if mel:
    ckpts["mejor-mel"] = min(mel, key=lambda p: float(p.split("val_mel=")[1].removesuffix(".ckpt")))
EXPORT = RAIZ / "exportado"
EXPORT.mkdir(exist_ok=True)
for etiqueta, ck in ckpts.items():
    onnx_path = EXPORT / etiqueta / f"es_MX-{NOMBRE}-medium.onnx"
    onnx_path.parent.mkdir(exist_ok=True)
    subprocess.run([sys.executable, "exportar_iurevoz.py", "--checkpoint", ck, "--output-file", str(onnx_path)],
                   cwd="/tmp/piper1-gpl", check=True)
    shutil.copy(CONFIG, onnx_path.with_suffix(".onnx.json"))
    print(etiqueta, "←", ck)

In [ ]:
# Muestras para escuchar (frases que no están en el corpus)
from piper import PiperVoice
from IPython.display import Audio, display
PRUEBA = [
    "Hola, ¿cómo estás? Hoy vamos a revisar el contrato de arrendamiento.",
    "Llámale a Guillermo para que traiga la llave del archivo.",
    "Ajá, órale, sí está bien, mándamelo cuando puedas.",
    "El total con iva es de mil ciento sesenta pesos.",
    "Según el artículo catorce constitucional, nadie puede ser privado de su libertad sin juicio.",
]
for etiqueta in ckpts:
    voz = PiperVoice.load(str(EXPORT / etiqueta / f"es_MX-{NOMBRE}-medium.onnx"))
    for k, t in enumerate(PRUEBA):
        destino = EXPORT / etiqueta / f"muestra-{k}.wav"
        with __import__("wave").open(str(destino), "wb") as w:
            voz.synthesize_wav(t, w)
        print(etiqueta, "·", t)
        display(Audio(str(destino)))

In [ ]:
# Limpieza para que la salida no pese de más: deja last.ckpt y el mejor por val_mel
todos = glob.glob(f"{ENTRENO}/**/*.ckpt", recursive=True)
conservar = set(ckpts.values())
for c in todos:
    if c not in conservar and not c.endswith("continuar.ckpt"):
        os.remove(c)
os.path.exists(ENTRENO / "continuar.ckpt") and os.remove(ENTRENO / "continuar.ckpt")
shutil.rmtree(RAIZ / "cache", ignore_errors=True)
shutil.rmtree(DATOS, ignore_errors=True)  # se regenera desde las entradas
print("checkpoints conservados:", *sorted(conservar), sep="\n  ")